# Bank Marketing - Model Experiments

## 1. Introduction

This notebook compares candidate models and tunes promising model families
to select a configuration for ranking term-deposit subscription records.

- Reproduce the sentinel-aware Logistic Regression baseline.
- Exclude `duration`, retain the other 19 prediction-time predictors, and
  preserve the established chronological split.
- Compare model families and tune hyperparameters using time-aware CV
  within Train.
- Fit the complete preprocessing and estimator Pipeline independently
  within each training fold.
- Compare Train-tuned candidates with the baseline on Validation, using
  AP as the primary metric and ROC-AUC and Top-5/10/20% metrics as
  supporting evidence.
- Consider temporal stability, inference cost, and model complexity
  alongside performance.
- Freeze the selected feature policy, preprocessing, estimator, parameters,
  and evaluation protocol before final evaluation.
- Keep Test reserved for final evaluation.

Training on Train+Validation and independent Test evaluation are performed
in the final evaluation notebook.

## 2. Data & Pipeline Setup


### 2.1 Load Data & Splits


Load the development rows in source order using the established chronological
boundaries: 28,831 Train rows and 8,238 Validation rows. Read only the first
37,069 rows so Test is not loaded. These boundaries are based on the full
dataset, not recalculated from the development subset.

Keep the 19 prediction-time predictors, excluding `duration` and target `y`.
Encode the target as `no = 0` and `yes = 1`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = (
    PROJECT_ROOT / "data" / "raw" / "bank-additional" / "bank-additional-full.csv"
)

# Established boundaries from the full 41,188-row dataset.
train_end = 28831
valid_end = 37069

df = pd.read_csv(RAW_DATA_PATH, sep=";", nrows=valid_end)
train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()

In [2]:
target = "y"
excluded_features = ["duration"]
modeling_features = [
    column for column in df.columns if column not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

target_mapping = {"no": 0, "yes": 1}
y_train = train_df[target].map(target_mapping)
y_valid = valid_df[target].map(target_mapping)

pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid)],
        "features": [X_train.shape[1], X_valid.shape[1]],
        "positive_rate": [y_train.mean(), y_valid.mean()],
    },
    index=["Train", "Validation"],
)

,rows,features,positive_rate
Train,28831,19,0.055704
Validation,8238,19,0.138626


The split summary provides context for later metric comparisons. Train and
Validation cover different periods, and their positive-class rates differ;
AP and Precision@K should be interpreted alongside these rates.


### 2.2 Sentinel-Aware Preprocessing


Use the same representation as the preprocessing and baseline notebooks:

| Feature group | Transformation |
|---|---|
| Eight ordinary numerical features | `StandardScaler` |
| `pdays` | Previous-contact indicator and elapsed days, then `StandardScaler` |
| Ten categorical features | `OneHotEncoder(handle_unknown="ignore")` |

For sentinel rows, `pdays = 999` becomes indicator `0` and elapsed days `0`.
Other rows receive indicator `1` and retain their elapsed days. Both columns
are needed to distinguish sentinel rows from an actual zero-day interval.
Literal `"unknown"` values remain categories.


In [3]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]


def split_pdays(values):
    pdays = np.asarray(values).reshape(-1)
    previously_contacted = (pdays != 999).astype(float)
    elapsed_days = np.where(pdays == 999, 0.0, pdays.astype(float))
    return np.column_stack([previously_contacted, elapsed_days])


def pdays_feature_names(transformer, input_features):
    return np.asarray(
        ["pdays_previously_contacted", "pdays_elapsed"],
        dtype=object,
    )

In [4]:
pdays_transformer = Pipeline(
    steps=[
        (
            "derive",
            FunctionTransformer(
                split_pdays,
                validate=False,
                feature_names_out=pdays_feature_names,
            ),
        ),
        ("scale", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        ("pdays", pdays_transformer, ["pdays"]),
    ],
    remainder="drop",
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

The complete Pipeline keeps preprocessing and the estimator together.
No fitting occurs during construction. Baseline fitting uses Train only;
later CV must fit a fresh complete Pipeline within each training fold.


## 3. Baseline Reproduction

### 3.1 Training & Validation Evaluation

Fit `LogisticRegression(max_iter=1000)` with the baseline preprocessing on
Train and generate unrounded positive-class scores for Validation.

Use AP as the primary metric and ROC-AUC as a supporting discrimination
metric. For Top-5/10/20%, set `K = ceil(n_records * fraction)` and sort scores
in descending order with a stable sort to preserve source order for ties.
Lift is relative to Validation prevalence.


In [5]:
baseline_pipeline.fit(X_train, y_train)
y_valid_score = baseline_pipeline.predict_proba(X_valid)[:, 1]

fitted_model = baseline_pipeline.named_steps["model"]
pd.Series(
    {
        "optimization_iterations": fitted_model.n_iter_[0],
        "transformed_features": fitted_model.n_features_in_,
        "validation_scores": len(y_valid_score),
    },
    name="value",
)

optimization_iterations      35
transformed_features         62
validation_scores          8238
Name: value, dtype: int64

In [6]:
validation_prevalence = y_valid.mean()
baseline_ap = average_precision_score(y_valid, y_valid_score)
baseline_roc_auc = roc_auc_score(y_valid, y_valid_score)

general_results = pd.Series(
    {
        "validation_prevalence": validation_prevalence,
        "average_precision": baseline_ap,
        "roc_auc": baseline_roc_auc,
    },
    name="Logistic Regression",
)

general_results.to_frame().round(6)

,Logistic Regression
validation_prevalence,0.138626
average_precision,0.288234
roc_auc,0.688023


In [7]:
ranked_indices = np.argsort(-y_valid_score, kind="stable")
ranked_y = y_valid.to_numpy()[ranked_indices]
cumulative_positives = np.cumsum(ranked_y)
total_positives = int(y_valid.sum())

ranking_rows = []
for fraction in [0.05, 0.10, 0.20]:
    k = int(np.ceil(len(y_valid) * fraction))
    captured = int(cumulative_positives[k - 1])
    precision = captured / k

    ranking_rows.append(
        {
            "top_fraction": fraction,
            "k": k,
            "captured_subscribers": captured,
            "precision_at_k": precision,
            "recall_at_k": captured / total_positives,
            "lift_at_k": precision / validation_prevalence,
        }
    )

ranking_results = pd.DataFrame(ranking_rows)
ranking_results.round(6)

,top_fraction,k,captured_subscribers,precision_at_k,recall_at_k,lift_at_k
0,0.05,412,179,0.434466,0.156743,3.134090
1,0.10,824,309,0.375000,0.270578,2.705123
2,0.20,1648,499,0.302791,0.436953,2.184233


Top-K metrics describe record-based ranking at different capacities, not
unique customers or daily contact quotas. Scores are not verified subscription
probabilities, and predictive metrics do not establish profitability or the
causal effect of contact.


### 3.2 Comparison with Stored Baseline Results

The reference values below are saved Validation results from
[Baseline Modeling](03-baseline-modeling.ipynb), rounded to six decimals.
They are historical reference values, not results newly verified here.

Run the preceding cells to populate the current results and their differences
from the reference. Small differences can reflect reference rounding; larger
differences should be investigated before interpreting candidate improvements.


In [8]:
stored_baseline = pd.Series(
    {
        "average_precision": 0.288234,
        "roc_auc": 0.688023,
        "precision_at_10pct": 0.375000,
        "recall_at_10pct": 0.270578,
        "lift_at_10pct": 2.705123,
    },
    name="stored_reference",
)

top_10 = ranking_results.loc[ranking_results["top_fraction"] == 0.10].iloc[0]
current_baseline = pd.Series(
    {
        "average_precision": baseline_ap,
        "roc_auc": baseline_roc_auc,
        "precision_at_10pct": top_10["precision_at_k"],
        "recall_at_10pct": top_10["recall_at_k"],
        "lift_at_10pct": top_10["lift_at_k"],
    },
    name="current_run",
)

baseline_comparison = pd.concat([stored_baseline, current_baseline], axis=1)
baseline_comparison["difference"] = current_baseline - stored_baseline
baseline_comparison

,stored_reference,current_run,difference
average_precision,0.288234,0.288234,3.437315e-07
roc_auc,0.688023,0.688023,-2.462303e-08
precision_at_10pct,0.375000,0.375000,0.000000e+00
recall_at_10pct,0.270578,0.270578,-6.654991e-08
lift_at_10pct,2.705123,2.705123,-4.080560e-07


Fresh-kernel execution of all eight code cells on 2026-10-26 reproduced
the stored baseline metrics to six decimal places: AP `0.288234`, ROC-AUC
`0.688023`, and Lift@10% `2.705123`. All five reference metric differences
are below `5e-7`, consistent with rounding of the stored reference.

The baseline provides a reproducible reference for candidate comparisons;
it is not a new model improvement or a final selection. Model-family
comparison and tuning have not been implemented in this notebook.
Only Train and Validation rows were loaded; Test was excluded.

The earlier raw-`pdays` results remain historical evidence, not an eligible
representation or the current baseline under the sentinel-aware policy.


## 4. Model-Family Comparison

### 4.1 Hypotheses & Candidate Models

### 4.2 Time-Aware Cross-Validation

### 4.3 Results & Tuning Candidates

## 5. Hyperparameter Tuning

### 5.1 Hypotheses & Search Scope

### 5.2 Train-Only CV Search

### 5.3 Results & Candidate Configurations

## 6. Validation Comparison

### 6.1 AP, ROC-AUC & Top-K Metrics


### 6.2 Results & Interpretation


### 6.3 Temporal Stability & Limitations


## 7. Model Selection & Final Configuration


## 8. Experiment Summary